# A rain of grains: from lists to numpy arrays
## Lab 04 · ACTAM 2026

A **grain** is a very short sound: a few hundredths of a second. Thousands of grains scattered in time make a texture, like rain on a roof.

Today we build a rain of grains in plain Python and ask a practical question: **how long does it take to compute?** Then we meet **numpy**, a library that computes on whole arrays of numbers at once. We will render the same rain again, faster and with shorter code.

### Setup

In [1]:
SR = 48000

import random
from math import sin, exp, pi
from IPython.display import Audio

def play(wave, sample_rate=SR):
    return Audio(wave, rate=sample_rate, normalize=False, autoplay=False)

## A · A rain in plain Python

### A1 · One grain
**Core · Write a function**

Write `grain(f, duration=0.05, amp=0.05, tau=0.01)`: a sine wave of frequency `f`, multiplied by an exponential decay with time constant `tau`. Sample `i` is

`amp · sin(2π · f · i / SR) · exp(−i / SR / tau)`

Listen to a grain at 2000 Hz, then try other frequencies and decay times.

### A2 · A score of grains

Before making sound, we decide *what* to play: for each grain, when it starts, its frequency and its amplitude. Three lists hold this **score**. Fixing the random seed makes the same rain every time.

In [ ]:
random.seed(4)
n_grains = 2000
rain_s = 10

onsets = [random.uniform(0, rain_s) for _ in range(n_grains)]
freqs = [random.uniform(1000, 4000) for _ in range(n_grains)]
amps = [random.uniform(0.01, 0.05) for _ in range(n_grains)]

print(onsets[:3], freqs[:3], amps[:3])

### A3 · Render the rain
**Core · Write a function**

Write `render(onsets, freqs, amps, grain_s=0.05)`:

1. create an output list of zeros, long enough for the last grain;
2. for each grain, compute its samples with `grain(...)`;
3. add them into the output, starting at sample `round(onset * SR)`.

`zip` walks the three lists together. `out[k] += v` is short for `out[k] = out[k] + v`.

In [ ]:
# your code here

## B · How long does it take?

In a notebook, a line that starts with `%` is a **special command**: an instruction for Jupyter, not Python.

- `%time` runs a statement once and reports how long it took. *Wall time* is the time on the clock.
- `%timeit` runs a statement many times and reports a typical time. Use it for short operations, whose single runs are too noisy to measure.

### B1 · Time the rain
**Core · Measure**

In [ ]:
%time rain = render(onsets, freqs, amps)
print(len(rain) / SR, "seconds of audio")

Half a second: not much. But a sound is rarely right at the first try.

### B2 · A denser rain
**Core · Predict, then measure**

Make the rain ten times denser: 20 000 grains over the same 10 seconds. Predict the computing time before measuring it.

In [ ]:
# your code here

Ten times the grains, about ten times the time: the cost grows with the number of samples we compute.

### Should we care?

Usually not at first. Clear, correct code comes first, and many renders take a fraction of a second. Measure before optimising: the slow part is often not the one we expect.

But looking for a sound means changing a parameter and listening again, dozens of times. Half a second per try is fine; several seconds per try slows the exploration down. Speed also matters when we play live, for long pieces, and on limited hardware such as a browser or a small board.

## C · Meet numpy

### What numpy is

**numpy** (“numerical Python”) is a free, open-source library for computing with numbers. It appeared in 2006, joining two earlier projects. Today almost all scientific Python is built on top of it: plotting with matplotlib, signal processing with scipy, data tables with pandas, machine learning, audio analysis.

It is not part of Python itself: it is installed separately, with `pip install numpy`. It is already available in our environment and in JupyterLite.

Its central object is the **array**: a sequence of numbers, like a list, with two differences:

- all its elements have **the same type**, for example 64-bit decimal numbers;
- they sit **side by side in one block of memory**, as bare numbers.

A Python list is more flexible. Each element is a separate Python object, of any type, and the list points to it. Flexibility has a cost: for every operation, Python visits each object, checks its type and creates a new object for the result. A numpy operation instead runs a loop written in C over the bare numbers.

Applying one operation to a whole array, instead of element by element, is called **vectorisation**.

By convention, numpy is imported as `np`.

In [ ]:
import numpy as np

samples = np.array([0.0, 0.5, -0.5, 0.25])
print(samples, samples.dtype, samples.shape, len(samples))

`dtype` is the type shared by all elements: `float64`, a 64-bit decimal number. `shape` gives the length along each dimension: here one dimension, 4 elements.

### C0 · One type for all
**Core · Predict, then run**

What type will numpy choose for each array?

In [ ]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2.5, 3]).dtype)
print(np.array([1, "two", 3]))
print([1, "two", 3.0])   # a list keeps each element as it is

numpy picks one type that fits every element: with a single decimal number, the integers become decimals too. With a string, everything becomes text, and arithmetic no longer works. For sound we always use decimal numbers.

### C1 · Same symbols, different meaning
**Core · Predict, then run**

Predict each result before running it.

In [ ]:
a_list = [1, 2, 3]
a_array = np.array([1, 2, 3])

print(a_list + a_list)
print(a_array + a_array)
print(a_list * 3)
print(a_array * 3)

On lists, `+` **joins** and `* 3` **repeats**: that is how we made `[0.0] * end`. On arrays, `+` **adds element by element** and `* 3` **multiplies every element**. Same symbols, different operations: check whether you have a list or an array.

### C2 · Arithmetic on whole arrays
**Core · Try**

With `x = np.array([1.0, 2.0, 3.0])` and `y = np.array([10.0, 20.0, 30.0])`, compute `x * y`, `0.5 * x`, `x + 1` and `x * y + 1`. A single number combined with an array acts on every element: this is called **broadcasting**.

What happens with `x + np.array([1.0, 2.0])`?

In [ ]:
# your code here

Arrays of different lengths cannot be combined element by element: numpy stops with an error.

### C3 · Making arrays, and functions on them
**Core · Read and run**

- `np.zeros(n)` gives `n` zeros: silence.
- `np.arange(n)` gives `0, 1, …, n-1`: sample indices. Divided by `SR`, they become the time of each sample.
- `np.sin`, `np.exp` and `np.abs` work on every element. Arrays also have summaries: `.max()`, `.min()`, `.sum()`.

In [ ]:
n = 5
print(np.zeros(n))
print(np.arange(n))
print(np.arange(n) / SR)
print(np.sin(np.array([0, np.pi / 2, np.pi])))
print(np.abs(np.array([0.1, -0.7, 0.3])).max())

### C4 · A slice is a view
**Core · Predict, then run**

We will add each grain into a slice of the output. Slicing a list makes a copy; slicing an array gives a **view** of the same memory. What does `out` contain at the end?

In [ ]:
out = np.zeros(8)
out[2:5] += np.array([1.0, 2.0, 3.0])
print(out)

part = out[2:5]
part[:] = 0
print(out)

## D · The rain with numpy

### D1 · One grain as an array
**Core · Rewrite**

Write `np_grain(f, duration=0.05, amp=0.05, tau=0.01)`. Start from the time of each sample, `t = np.arange(n) / SR`: the whole grain is then one expression.

Check it against `grain` with `np.allclose`, which is true when two sequences match up to rounding. Time both with `%timeit`.

In [ ]:
# your code here

### D2 · Add each grain with a slice
**Core · Rewrite**

Write `np_render(onsets, freqs, amps, grain_s=0.05)`. Start from `np.zeros(end)`, and replace the inner loop with one line: `out[start:start + len(g)] += g`.

Render the same score, check it against `rain` and time it.

In [ ]:
# your code here

The numpy version describes whole grains instead of single samples: “a sine times a decay”, “add this grain here”. It is shorter, and on our computer it ran about 15 times faster.

### D3 · The dense rain, again
**Core · Measure**

Render the 20 000-grain score with `np_render`. Compare the time with B2.

In [ ]:
# your code here

## E · Extensions

### E1 · All the grains at once
**Extension · Read, then try**

Every grain has the same length, so all of them fit in one **2D array**: one row per grain, one column per sample.

- `freqs[:, None]` turns the frequencies into a column. Multiplied by the row `t`, broadcasting gives every frequency at every time: a table of shape `(number of grains, samples per grain)`.
- `np.add.at(out, positions, values)` adds every value at its position, even when positions repeat (grains overlap).

Compare the result and the time with `np_render`. Is the fully vectorised version always faster?

In [ ]:
# your code here

On our computer this version was slightly *slower* than `np_render`. `np.add.at` must handle positions that repeat, and that costs time. Vectorising everything is not automatically faster: measure. Here the simple loop over grains, each added with one slice, is already short and fast.

### E2 · A musical rain
**Extension · Try**

numpy has its own random generator: `rng = np.random.default_rng(seed)`. `rng.uniform(low, high, n)` gives `n` values at once, and `rng.choice(values, n)` picks `n` of them.

Draw the frequencies from a few notes of a scale instead of a continuous range, and let the rain grow denser over time: the square root of a uniform draw puts more onsets towards the end.

In [ ]:
# your code here

### E3 · A rain in stereo
**Extension · Try**

A stereo sound is an array of shape `(2, n)`: the left channel in row 0, the right in row 1. `play` plays it as stereo.

Give each grain a random position between left and right: its amplitude on the left is `amp * (1 - pan)` and on the right `amp * pan`.

In [ ]:
# your code here